In [49]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# =========================
# 1. LOAD DATA
# =========================
df = pd.read_csv("Dataset_14-day_AA_depression_symptoms_mood_and_PHQ-9.csv")
df.columns = df.columns.str.strip()

In [50]:
# 2. DEFINE COLUMNS
# =========================
phq_cols = [f"phq{i}" for i in range(1, 10)]

q_cols = [
    "q1","q2","q3","q4","q5","q6","q7","q8","q9",
    "q10","q11","q12","q13","q14","q16","q46","q47"
]

In [51]:
# 3. CLEAN PHQ (ONLY FOR TARGET)
# =========================
for col in phq_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    df[col] = df[col].fillna(df[col].median())

# PHQ total score
df["phq_total"] = df[phq_cols].sum(axis=1)

In [52]:
# Target labels
def label(x):
    if x <= 4:
        return 0
    elif x <= 9:
        return 1
    elif x <= 14:
        return 2
    else:
        return 3

df["depression_target"] = df["phq_total"].apply(label)

In [53]:
# 4. CLEAN Q FEATURES
# =========================
for col in q_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    df[col] = df[col].fillna(df[col].median())

In [54]:
# 5. OTHER FEATURES
# =========================

# happiness
df["happiness.score"] = pd.to_numeric(df["happiness.score"], errors="coerce")
df["happiness.score"] = df["happiness.score"].fillna(df["happiness.score"].median())

# age
df["age"] = pd.to_numeric(df["age"], errors="coerce")
df["age"] = df["age"].fillna(df["age"].median())

# sex encoding
df["sex"] = df["sex"].fillna(df["sex"].mode()[0])
df["sex"] = df["sex"].map({"male": 0, "female": 1})
df["sex"] = df["sex"].fillna(0)

In [55]:
# 6. TIME FEATURES
# =========================
df["time"] = pd.to_datetime(df["time"], errors="coerce")
df = df.sort_values(["user_id", "time"])

df["hour"] = df["time"].dt.hour.fillna(0)

In [56]:
# 7. SIMPLE BEHAVIOR FEATURES (KEEP ONLY USEFUL ONES)
# =========================
df["prev_happiness"] = df.groupby("user_id")["happiness.score"].shift(1)
df["prev_happiness"] = df["prev_happiness"].fillna(df["happiness.score"])

# =========================
# 8. ENCODE PERIOD
# =========================
df = pd.get_dummies(df, columns=["period.name"], dtype=int)

In [57]:
# 9. FINAL FEATURES (NO LEAKAGE)
# =========================

drop_cols = (
    phq_cols +              #  remove leakage
    ["phq_total", "depression_target", "user_id", "time"]
)

X = df.drop(columns=[col for col in drop_cols if col in df.columns])
y = df["depression_target"]


In [58]:
X = X.select_dtypes(include=["number"])

In [59]:
X = X.drop(columns=["id", "Unnamed: 0"])

In [60]:
# KEEP ONLY NUMERIC FEATURES
X = X.select_dtypes(include=["number"])

# SPLIT
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# MODEL
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# EVALUATION
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.8040247678018576
              precision    recall  f1-score   support

           0       0.89      0.86      0.87       195
           1       0.67      0.51      0.58       440
           2       0.78      0.66      0.72       706
           3       0.82      0.92      0.87      1889

    accuracy                           0.80      3230
   macro avg       0.79      0.74      0.76      3230
weighted avg       0.80      0.80      0.80      3230



In [61]:
# 13. FEATURE IMPORTANCE
# =========================
import pandas as pd

importance = pd.Series(
    model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print("\nTop Features:\n")
print(importance.head(15))


Top Features:

age                0.251565
phq.day            0.204003
hour               0.090506
prev_happiness     0.068569
happiness.score    0.065801
sex                0.033840
q13                0.018697
q10                0.017673
q16                0.016921
q9                 0.016778
q11                0.015713
q47                0.015463
q46                0.015455
q6                 0.015215
q5                 0.015198
dtype: float64


In [62]:
for col in X.columns:
    print(col)

age
sex
q1
q2
q3
q4
q5
q6
q7
q8
q9
q10
q11
q12
q13
q14
q16
q46
q47
happiness.score
phq.day
hour
prev_happiness
period.name_evening
period.name_midday
period.name_morning
